        # 📈 L04　線性迴歸
        **統計冒險之旅 2026**　｜　Day 2（09/21 一）⛰️ 模型之嶺　｜　關卡　｜　🏅 100 XP

        📖 ISLP Ch3；資料：Advertising（模擬）、勇者咖啡每日營收


        ### 🎯 這一關你會學到
        - 相關係數與散佈圖；最小平方法
- LinearRegression：係數怎麼讀、R² 與 RMSE
- 多元迴歸、one-hot 編碼、statsmodels 看 p 值

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L04"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["4-1", "4-2", "4-3", "4-4", "4-5", "4-6"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_4_1(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "相關矩陣"), 列=4, 欄=4)
    if not ok: return (False, msg)
    if not 約等於(抓變數(ns, "TV相關"), 0.81808, 0.005): return (False, "TV相關 = 相關矩陣.loc['TV', 'Sales']。")
    return (str(抓變數(ns, "最相關")) == "TV", "最相關 用 .idxmax() 找出相關最高的欄位。")
任務定義("4-1", _check_4_1, 提示=".idxmax() 回傳最大值的索引名稱。")

def _check_4_2(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "截距"), 6.46507, 0.01): return (False, "截距 = model.intercept_。")
    if not 約等於(抓變數(ns, "斜率"), 0.051462, 0.0005): return (False, "斜率 = model.coef_[0]。")
    if not 約等於(抓變數(ns, "R2"), 0.66925, 0.005): return (False, "R2 = model.score(X, y)。")
    return (約等於(抓變數(ns, "預測_TV100"), 11.6113, 0.05), "預測_TV100 = 截距 + 斜率 × 100。")
任務定義("4-2", _check_4_2, 提示="model.coef_ 是陣列，第一個元素是 TV 的斜率。")

def _check_4_3(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "RMSE"), 3.12301, 0.01): return (False, "RMSE = np.sqrt(mean_squared_error(y, pred))。")
    return (約等於(抓變數(ns, "誤差量級"), 3.12, 0.011), "誤差量級 = round(RMSE, 2)。")
任務定義("4-3", _check_4_3, 提示="mean_squared_error 是平方誤差的平均，開根號才是 RMSE。")

def _check_4_4(run):
    out, ns = run()
    s = 抓變數(ns, "係數")
    ok, msg = 資料框像(s, 列=3, 含欄位=["TV", "Radio", "Newspaper"], 種類="Series")
    if not ok: return (False, msg)
    if not 約等於(s["Radio"], 0.18278, 0.002): return (False, "係數 不對，三個欄位一起 fit。")
    if not 約等於(抓變數(ns, "R2_3"), 0.89324, 0.005): return (False, "R2_3 = model3.score(X3, y)。")
    return (str(抓變數(ns, "最沒用")) == "Newspaper", "最沒用 = 係數.abs().idxmin()。")
任務定義("4-4", _check_4_4, 提示=".abs().idxmin() 找絕對值最小的那一項。")

def _check_4_5(run):
    out, ns = run()
    p = 抓變數(ns, "p值們")
    if "Newspaper" not in list(getattr(p, "index", [])): return (False, "p值們 = result.pvalues。")
    if not 約等於(p["Newspaper"], 0.43680, 0.01): return (False, "Newspaper 的 p 值不對。")
    return (str(抓變數(ns, "不顯著的變數")) == "Newspaper", "p > 0.05 的是哪個變數？")
任務定義("4-5", _check_4_5, 提示="p值們[p值們 > 0.05].index[0]")

def _check_4_6(run):
    out, ns = run()
    if not 約等於(抓變數(ns, "促銷係數"), 1181.281, 1.0): return (False, "促銷係數 = 係數d['促銷']。")
    if not 約等於(抓變數(ns, "信義店係數"), 1360.297, 1.0): return (False, "信義店係數 = 係數d['分店_信義店']。")
    return (約等於(抓變數(ns, "R2_d"), 0.14929, 0.005), "R2_d = md.score(X, daily['營收'])。")
任務定義("4-6", _check_4_6, 提示="one-hot 之後的欄名是 分店_信義店、分店_板橋店。")

## ⛰️ 4-1　機器學習是什麼？
Day 1 我們問「這是真的還是巧合」；Day 2 開始問「**給我 X，你能預測 Y 嗎？**」。
所有機器學習都可以寫成一條式子：

> **Y = f(X) + ε**　　（結果 ＝ 規則(線索) ＋ 運氣）

- **X**：線索（特徵 feature）——廣告預算、天氣、客人的來店次數。
- **Y**：想預測的東西（目標 target）——銷售量、營收、會不會回購。
- **f**：藏在資料裡的規則，**模型就是我們對 f 的猜測**。
- **ε**：怎麼樣都猜不到的隨機部分（不可縮減的誤差）。

| 兩種目的 | 問的問題 | 例子 |
|---|---|---|
| **預測** | Y 會是多少？ | 下週營收、這位病人的風險 |
| **推論** | 哪些 X 重要？影響多大？ | 電視廣告每多花一萬元多賣幾件 |

有標準答案 Y 可以學的叫**監督式學習**（Day 2、Day 3 前半）；沒有 Y、只能找結構的叫**非監督式學習**（Day 3 的分群）。
scikit-learn 把所有模型都做成同樣的三步驟：`model.fit(X, y)` 學 → `model.predict(X)` 猜 → `model.score(X, y)` 評分。

In [ ]:
import pandas as pd, numpy as np
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
adv = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0/data/advertising.csv")     # 200 個市場：電視／廣播／報紙廣告預算（千元）與銷售量（千件）
adv.head()

## 4-2　從相關到迴歸
**相關係數 r**（−1 ～ 1）量「兩個數字一起變動的程度」：r 越接近 1，散佈圖越像一條往右上的線。
> ⚠️ **相關不等於因果**：冰淇淋銷量和溺水人數一起上升，是因為天氣熱。報紙廣告和銷售看起來有關，可能只是「花得起報紙廣告的市場，電視廣告也花得多」在蹭熱度——多元迴歸（4-4）會把它揪出來。

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

In [ ]:
print(adv.corr().round(3))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for i, col in enumerate(["TV", "Radio", "Newspaper"]):
    ax[i].scatter(adv[col], adv["Sales"], s=12, alpha=.6); ax[i].set_xlabel(col); ax[i].set_ylabel("Sales")
    ax[i].set_title(f"{col} vs Sales（r = {adv[col].corr(adv['Sales']):.2f}）")
plt.tight_layout(); plt.show()

## 4-3　最小平方法：彈珠滾到碗底
線性迴歸找一條直線 **Sales = 截距 + 斜率 × TV**，讓所有點到直線的**垂直距離平方和**最小——像彈珠在碗裡滾到最低點。
- **斜率**：TV 每多 1（千元），Sales 多幾（千件）。
- **R²**：這條線解釋掉幾成的 Sales 變化（0～1）。
- **RMSE**：誤差的尺度／量級，單位和 Y 一樣；因為先平方，所以比 MAE 更重懲罰大誤差。它不是「每筆通常猜錯多少」的直接估計。

In [ ]:
model = LinearRegression().fit(adv[["TV"]], adv["Sales"])          # X 要是「表格」（兩層中括號）
print("截距", round(model.intercept_, 3), "斜率", round(model.coef_[0], 4))
print("TV 花 100 → 預測 Sales", model.predict(pd.DataFrame({"TV": [100]})).round(2))
pred = model.predict(adv[["TV"]])
print("R² =", round(r2_score(adv["Sales"], pred), 3), "| RMSE =", round(np.sqrt(mean_squared_error(adv["Sales"], pred)), 3))
plt.scatter(adv["TV"], adv["Sales"], s=12, alpha=.5); plt.plot(adv["TV"], pred, color="red"); plt.title("最小平方直線"); plt.xlabel("TV"); plt.ylabel("Sales"); plt.show()

## 4-4　多元迴歸與 p 值：報紙廣告有用嗎？
把三種廣告一起放進去：**Sales = b0 + b1·TV + b2·Radio + b3·Newspaper**。每個係數的意思是「**其他不變**，這一項每多 1，Sales 多幾」。
`statsmodels` 會多給你每個係數的 **p 值**（Day 1 學過）：p 很小 → 這個變數的效果「不像巧合」。

In [ ]:
import statsmodels.formula.api as smf
model3 = LinearRegression().fit(adv[["TV", "Radio", "Newspaper"]], adv["Sales"])
print(pd.Series(model3.coef_, index=["TV", "Radio", "Newspaper"]).round(4))
print("R² =", round(model3.score(adv[["TV", "Radio", "Newspaper"]], adv["Sales"]), 3))
result = smf.ols("Sales ~ TV + Radio + Newspaper", data=adv).fit()
print(result.summary().tables[1])          # coef、p 值（P>|t|）都在這張表

## 4-5　類別變數：one-hot 編碼
模型只吃數字。「分店」這種文字要先變成 0／1 欄位（**one-hot**）：`pd.get_dummies(..., drop_first=True)` 會把中壢店當基準，多出「分店_信義店」「分店_板橋店」兩欄，它們的係數就是「和中壢店比，多賣多少」。
資料：勇者咖啡每日營收 `https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0/data/coffee_daily.csv`

In [ ]:
daily = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0/data/coffee_daily.csv")
X = pd.get_dummies(daily[["氣溫", "促銷", "分店"]], columns=["分店"], drop_first=True).astype(float)
print(X.head(3))
md = LinearRegression().fit(X, daily["營收"])
print(pd.Series(md.coef_, index=X.columns).round(1))
print("截距", round(md.intercept_, 1), "R²", round(md.score(X, daily["營收"]), 3), "（只用三個線索，R² 不高很正常；Day 3 會加更多）")

### 🎯 任務 4-1　相關係數

算出 `相關矩陣`（`adv.corr()`），取出 `TV相關`（TV 與 Sales 的相關係數），並把三種廣告中與 Sales 最相關的欄位名稱存成 `最相關`。

In [ ]:
# 🎯 任務 4-1　相關係數（請保留這一行）
相關矩陣 = adv.corr()
TV相關 = 相關矩陣.loc["TV", "Sales"]
最相關 = 相關矩陣.loc[["TV", "Radio", "Newspaper"], "Sales"].???()
print(round(TV相關, 3), 最相關)

In [ ]:
檢查("4-1")   # ◀ 執行這一格，看看任務 4-1 有沒有過關

### 🎯 任務 4-2　簡單線性迴歸

用 TV 預測 Sales：建立 `model`（`LinearRegression().fit(adv[['TV']], adv['Sales'])`），取出 `截距`、`斜率`、`R2`（`model.score`），並算 `預測_TV100`（TV = 100 時的預測值）。

In [ ]:
# 🎯 任務 4-2　簡單線性迴歸（請保留這一行）
model = LinearRegression().fit(adv[["TV"]], adv["Sales"])
截距 = model.intercept_
斜率 = ???
R2 = model.score(adv[["TV"]], adv["Sales"])
預測_TV100 = model.predict(pd.DataFrame({"TV": [100]}))[0]
print(round(截距, 3), round(斜率, 4), round(R2, 3), round(預測_TV100, 2))

In [ ]:
檢查("4-2")   # ◀ 執行這一格，看看任務 4-2 有沒有過關

### 🎯 任務 4-3　RMSE

用 `model` 對全部資料預測存成 `pred`，算出 `RMSE`（`np.sqrt(mean_squared_error(adv['Sales'], pred))`），並把「誤差尺度／量級（千件）」四捨五入到 2 位存成 `誤差量級`。RMSE 會較重懲罰大誤差，不要把它說成每筆的典型誤差。

In [ ]:
# 🎯 任務 4-3　RMSE（請保留這一行）
pred = model.predict(adv[["TV"]])
RMSE = ???
誤差量級 = round(RMSE, 2)
print(RMSE, 誤差量級)

In [ ]:
檢查("4-3")   # ◀ 執行這一格，看看任務 4-3 有沒有過關

### 🎯 任務 4-4　多元迴歸

用 TV、Radio、Newspaper 一起預測 Sales：建立 `model3`，把係數做成 Series `係數`（索引為三個欄位名），並算 `R2_3`。哪一個係數最接近 0？存成 `最沒用`。

In [ ]:
# 🎯 任務 4-4　多元迴歸（請保留這一行）
X3 = adv[["TV", "Radio", "Newspaper"]]
model3 = LinearRegression().fit(X3, adv["Sales"])
係數 = pd.Series(model3.coef_, index=X3.columns)
R2_3 = ???
最沒用 = 係數.abs().???()
print(係數.round(4)); print(round(R2_3, 3), 最沒用)

In [ ]:
檢查("4-4")   # ◀ 執行這一格，看看任務 4-4 有沒有過關

### 🎯 任務 4-5　看 p 值

用 statsmodels 跑 `Sales ~ TV + Radio + Newspaper`，把 `result.pvalues` 存成 `p值們`，並把 p 值 > 0.05 的變數名稱存成 `不顯著的變數`（字串）。

In [ ]:
# 🎯 任務 4-5　看 p 值（請保留這一行）
import statsmodels.formula.api as smf
result = smf.ols("Sales ~ TV + Radio + Newspaper", data=adv).fit()
p值們 = ???
不顯著的變數 = ???
print(p值們.round(4)); print(不顯著的變數)

In [ ]:
檢查("4-5")   # ◀ 執行這一格，看看任務 4-5 有沒有過關

### 🎯 任務 4-6　類別變數 one-hot

用勇者咖啡每日營收：`X = pd.get_dummies(daily[['氣溫', '促銷', '分店']], columns=['分店'], drop_first=True).astype(float)`，建立 `md` 預測 `營收`，取出 `促銷係數`（促銷那一欄的係數）與 `信義店係數`（和中壢店比多賣多少），以及 `R2_d`。

In [ ]:
# 🎯 任務 4-6　類別變數 one-hot（請保留這一行）
daily = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.1.0/data/coffee_daily.csv")
X = pd.get_dummies(daily[["氣溫", "促銷", "分店"]], columns=["分店"], drop_first=True).astype(float)
md = LinearRegression().fit(X, daily["營收"])
係數d = pd.Series(md.coef_, index=X.columns)
促銷係數 = ???
信義店係數 = ???
R2_d = ???
print(係數d.round(1)); print(round(促銷係數, 1), round(信義店係數, 1), round(R2_d, 3))

In [ ]:
檢查("4-6")   # ◀ 執行這一格，看看任務 4-6 有沒有過關

## 🌟 進階挑戰（不計分）
1. 在 4-6 加入 `星期` 的 one-hot，R² 提高多少？週六的係數是多少？
2. 在 Advertising 加入 `TV * Radio` 的交互作用項（`adv['TV'] * adv['Radio']`），R² 會跳到多少？（電視和廣播一起打廣告有加乘效果）

---
## 🔑 通關密語
　你已經會用一條直線做預測，也會讀係數和 p 值了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🎯 L05 過度擬合與偏差－變異** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.1.0/notebooks/L05_overfitting.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/